In [35]:
import yaml
from pathlib import Path
import pandas as pd

In [44]:
config_path = Path.cwd().parent / "config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

root = Path(config['project_root'])
base_path = root / "data" /"raw"
print(f"Loading data from: {base_path}")

Loading data from: C:\AI-Based Smart City Air Quality Monitoring and Forecasting System for Greater Bilbao\data\raw


In [45]:
all_dfs = []

for year_folder in sorted(base_path.iterdir()):
    if year_folder.is_dir():

        year = int(year_folder.name)

        for csv_file in year_folder.glob("*.csv"):

            station = csv_file.stem.upper()

            df = pd.read_csv(
                csv_file,
                encoding="latin1",
                sep=";"
            )

            df["year"] = year
            df["station"] = station

            all_dfs.append(df)

final_df = pd.concat(all_dfs, ignore_index=True)
final_df

,Date,NO (µg/m3),NO2 (µg/m3),NOX (µg/m3),O3 (µg/m3),SO2 (µg/m3),year,station,CO (mg/m3),PM10 (µg/m3),"PM2,5 (µg/m3)",Benceno (µg/m3),Ortoxileno (µg/m3),Tolueno (µg/m3),NH3 (µg/m3),SH2 (µg/m3),M-P-XILENO (µg/m3),CO 8h (mg/m3),Etilbenceno (µg/m3),O3 8h (µg/m3)
0,31/12/2012,2.0,6.0,9.0,80.0,14.0,2012,ABANTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30/12/2012,2.0,6.0,8.0,69.0,9.0,2012,ABANTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,29/12/2012,2.0,9.0,11.0,76.0,10.0,2012,ABANTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28/12/2012,5.0,14.0,22.0,67.0,14.0,2012,ABANTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,27/12/2012,2.0,11.0,14.0,58.0,18.0,2012,ABANTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52253,05/01/2026,3,26.0,31.0,NaN,4.0,2026,SANTURCE,NaN,8.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52254,04/01/2026,1,16.0,18.0,NaN,4.0,2026,SANTURCE,NaN,6.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52255,03/01/2026,1,11.0,12.0,NaN,3.0,2026,SANTURCE,NaN,4.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52256,02/01/2026,7,25.0,36.0,NaN,5.0,2026,SANTURCE,NaN,8.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
metadata_folder = Path(config['metadata_path'])
stations_file = metadata_folder / "estaciones2026.xlsx"

In [47]:
stations = pd.read_excel(r"C:\AI-Based Smart City Air Quality Monitoring and Forecasting System for Greater Bilbao\data\metadata\estaciones2026.xlsx")

stations = stations[
    ["Name", "Province", "Town", "Address", "Latitude", "Longitude"]
].copy()

stations["station"] = (
    stations["Name"]
    .str.upper()
    .str.replace(r"\s*\(.*\)", "", regex=True)  
    .str.strip()
)

for col in ["Latitude", "Longitude"]:
    stations[col] = (
        stations[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )
    stations[col] = pd.to_numeric(stations[col], errors="coerce")

stations = stations.drop(columns=["Name"])
stations

,Province,Town,Address,Latitude,Longitude,station
0,Bizkaia,Abanto y Ciérvana-Abanto Zierbena,"Avda. del Minero, 2. Ayuntamiento",43.320474,-3.074156,ABANTO
1,Bizkaia,Getxo,"Carretera de Galea, s/n",43.362056,-3.022782,ALGORTA_BBIZI2
2,Bizkaia,Alonsotegi,"C/ Baztieta, s/n. Bº Irauregi",43.247568,-2.988024,ALONSOTEGI
3,Bizkaia,Barakaldo,"C/ Hogar propio, 7. CIFP Nicolás Larburu",43.298379,-2.987133,BARAKALDO
4,Bizkaia,Basauri,"C/ Uribarri, 5. CEIP Bizkotxalde.",43.241131,-2.883761,BASAURI
5,Bizkaia,Erandio,"Avda. José Luis Goyoaga, s/n (Pasaje de Altzaga)",43.302653,-2.977240,ERANDIO
6,Bizkaia,Bilbao,"C/ Alameda Mazarredo, s/n (Guggenheim)",43.267506,-2.935188,MAZARREDO
7,Bizkaia,Muskiz,"C/ Giba Fregenal, s/n. Estación de Renfe",43.320713,-3.112716,MUSKIZ
8,Bizkaia,Sondika,"C/ Iturrikosolo, s/n",43.298421,-2.930386,SANGRONIZ
9,Bizkaia,Santurtzi,"C/ Vista Alegre, 29",43.333012,-3.042560,SANTURCE


In [48]:
final_df["station"] = final_df["station"].str.upper().str.strip()

final_df = final_df.merge(
    stations,
    on="station",
    how="left"
)

print(final_df.shape)
final_df

(52258, 25)


,Date,NO (µg/m3),NO2 (µg/m3),NOX (µg/m3),O3 (µg/m3),SO2 (µg/m3),year,station,CO (mg/m3),PM10 (µg/m3),...,SH2 (µg/m3),M-P-XILENO (µg/m3),CO 8h (mg/m3),Etilbenceno (µg/m3),O3 8h (µg/m3),Province,Town,Address,Latitude,Longitude
0,31/12/2012,2.0,6.0,9.0,80.0,14.0,2012,ABANTO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bizkaia,Abanto y Ciérvana-Abanto Zierbena,"Avda. del Minero, 2. Ayuntamiento",43.320474,-3.074156
1,30/12/2012,2.0,6.0,8.0,69.0,9.0,2012,ABANTO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bizkaia,Abanto y Ciérvana-Abanto Zierbena,"Avda. del Minero, 2. Ayuntamiento",43.320474,-3.074156
2,29/12/2012,2.0,9.0,11.0,76.0,10.0,2012,ABANTO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bizkaia,Abanto y Ciérvana-Abanto Zierbena,"Avda. del Minero, 2. Ayuntamiento",43.320474,-3.074156
3,28/12/2012,5.0,14.0,22.0,67.0,14.0,2012,ABANTO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bizkaia,Abanto y Ciérvana-Abanto Zierbena,"Avda. del Minero, 2. Ayuntamiento",43.320474,-3.074156
4,27/12/2012,2.0,11.0,14.0,58.0,18.0,2012,ABANTO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bizkaia,Abanto y Ciérvana-Abanto Zierbena,"Avda. del Minero, 2. Ayuntamiento",43.320474,-3.074156
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52253,05/01/2026,3,26.0,31.0,NaN,4.0,2026,SANTURCE,NaN,8.0,...,NaN,NaN,NaN,NaN,NaN,Bizkaia,Santurtzi,"C/ Vista Alegre, 29",43.333012,-3.042560
52254,04/01/2026,1,16.0,18.0,NaN,4.0,2026,SANTURCE,NaN,6.0,...,NaN,NaN,NaN,NaN,NaN,Bizkaia,Santurtzi,"C/ Vista Alegre, 29",43.333012,-3.042560
52255,03/01/2026,1,11.0,12.0,NaN,3.0,2026,SANTURCE,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,Bizkaia,Santurtzi,"C/ Vista Alegre, 29",43.333012,-3.042560
52256,02/01/2026,7,25.0,36.0,NaN,5.0,2026,SANTURCE,NaN,8.0,...,NaN,NaN,NaN,NaN,NaN,Bizkaia,Santurtzi,"C/ Vista Alegre, 29",43.333012,-3.042560


In [49]:
output_path = root / "data" / "processed"

output_file = output_path / "air_quality_bilbao_2012_2026.csv"

final_df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"File saved successfully at: {output_file}")

File saved successfully at: C:\AI-Based Smart City Air Quality Monitoring and Forecasting System for Greater Bilbao\data\processed\air_quality_bilbao_2012_2026.csv
